In [0]:
# 02b — Silver: Demanda limpia + puente ruta→empresa→parada

import pandas as pd
from datetime import datetime, timezone

In [0]:
CATALOG = "gobierno_abierto"
SCHEMA_BRONZE = "a_bronze"
SCHEMA_SILVER = "b_silver"

In [0]:
 ## 1. Cortes de boleto — de formato ancho a largo
cortes_bronze = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.cortes_boleto_diario").toPandas()
print("Columnas originales:", list(cortes_bronze.columns))
cortes_bronze.head()

In [0]:
COLUMNAS_EMPRESA = ["AUCOR", "Coniferal", "ERSA", "TAMSE"]
columnas_empresa_presentes = [c for c in COLUMNAS_EMPRESA if c in cortes_bronze.columns]
print(f"Columnas de empresa detectadas: {columnas_empresa_presentes}")

faltantes = set(COLUMNAS_EMPRESA) - set(columnas_empresa_presentes)
if faltantes:
    print(f"No se encontraron estas columnas esperadas: {faltantes} — revisar nombres reales")

MESES_ES = {
    "Enero": 1, "Febrero": 2, "Marzo": 3, "Abril": 4, "Mayo": 5, "Junio": 6,
    "Julio": 7, "Agosto": 8, "Septiembre": 9, "Octubre": 10, "Noviembre": 11, "Diciembre": 12,
}

cortes_bronze["mes_num"] = cortes_bronze["Mes"].str.strip().map(MESES_ES)

sin_mapear = cortes_bronze["mes_num"].isna().sum()
if sin_mapear > 0:
    print(f"⚠️ {sin_mapear} filas con nombre de mes no reconocido: {cortes_bronze[cortes_bronze['mes_num'].isna()]['Mes'].unique()}")

cortes_bronze["fecha"] = pd.to_datetime(
    dict(year=cortes_bronze["Año"], month=cortes_bronze["mes_num"], day=cortes_bronze["Dia"]),
    errors="coerce",
)

antes = len(cortes_bronze)
cortes_bronze = cortes_bronze.dropna(subset=["fecha"])
print(f"Filas descartadas por fecha inválida: {antes - len(cortes_bronze)}")


In [0]:
cortes_largo = cortes_bronze.melt(
    id_vars=["fecha"],
    value_vars=columnas_empresa_presentes,
    var_name="empresa",
    value_name="cortes_boleto",
)
cortes_largo["cortes_boleto"] = pd.to_numeric(cortes_largo["cortes_boleto"], errors="coerce")
cortes_largo = cortes_largo.dropna(subset=["cortes_boleto"])

print(f"Filas en formato largo: {len(cortes_largo)}")
cortes_largo.head()


In [0]:
cortes_bronze_check = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.cortes_boleto_diario").toPandas()

print(f"Filas en Bronze: {len(cortes_bronze_check)}")
print(f"Columnas: {list(cortes_bronze_check.columns)}")
display(cortes_bronze_check.head(3))

In [0]:
# Promedio diario por empresa — métrica que se usa en Gold para repartir por barrio

promedio_diario_por_empresa = (
    cortes_largo.groupby("empresa")["cortes_boleto"].mean().rename("promedio_diario").reset_index()
)
print(promedio_diario_por_empresa)

cortes_silver = cortes_largo.copy()
cortes_silver["_processed_at"] = datetime.now(timezone.utc)

spark.createDataFrame(cortes_silver).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.cortes_boleto")

spark.createDataFrame(promedio_diario_por_empresa).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.cortes_boleto_promedio_empresa")

print(f"cortes_boleto: {len(cortes_silver)} filas escritas.")

In [0]:
## 2. Puente route_id → agency_name → stop_id

agency_pdf = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.gtfs_agency").toPandas()
routes_pdf = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.gtfs_routes").toPandas()
trips_pdf = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.gtfs_trips").toPandas()
stop_times_pdf = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.gtfs_stop_times").toPandas()

print(f"agency: {len(agency_pdf)} | routes: {len(routes_pdf)} | trips: {len(trips_pdf)} | stop_times: {len(stop_times_pdf)}")

# route_id -> agency_name
routes_con_agencia = routes_pdf.merge(
    agency_pdf[["agency_id", "agency_name"]], on="agency_id", how="left"
)

sin_agencia = routes_con_agencia["agency_name"].isna().sum()
print(f"Líneas sin agencia asociada (probablemente AUCOR u otras no incluidas en el GTFS): {sin_agencia}")

# trip_id -> route_id, y trip_id -> stop_id (vía stop_times), unidos en route_id -> stop_id
trip_route = trips_pdf[["trip_id", "route_id"]].drop_duplicates()
trip_stop = stop_times_pdf[["trip_id", "stop_id"]].drop_duplicates()

route_stops = trip_route.merge(trip_stop, on="trip_id", how="inner")[["route_id", "stop_id"]].drop_duplicates()
route_stops = route_stops.merge(
    routes_con_agencia[["route_id", "route_short_name", "agency_name"]], on="route_id", how="left"
)

print(f"Pares route_id-stop_id únicos: {len(route_stops)}")

sin_agencia_stops = route_stops["agency_name"].isna()
print(f"Pares sin agencia (se excluyen del cruce de demanda en Gold): {sin_agencia_stops.sum()}")
route_stops["_processed_at"] = datetime.now(timezone.utc)

In [0]:
## Verificación final

spark.createDataFrame(route_stops).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.route_stops")

count = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.route_stops").count()
print(f"route_stops: {count} filas escritas.")

for tabla in ["cortes_boleto", "cortes_boleto_promedio_empresa", "route_stops"]:
    count = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.{tabla}").count()
    print(f"{CATALOG}.{SCHEMA_SILVER}.{tabla}: {count} filas")